In [ ]:
"""
🛰️ 군사 이미지 분석 & 제로샷 분류 체험 스크립트 🛰️

✅ 데이터셋 이름: AMANMP0007/military-labeled-clip
✅ 의미: 군사 작전 영상(DVIDS)에서 추출한 객체 단위 크롭(Crops) 이미지와,
        해당 객체와 이미지를 설명하는 상세한 캡션(Caption)들이 포함된 데이터셋입니다.
✅ 용도: 이미지 분류(Image Classification) 및 '어떤 것이 이 객체인지'를 설명하는
       제로샷 이미지 분류(Zero-shot Image Classification) 능력을 학습시키는 데 최적화되어 있습니다.

✨ 친절한 튜터 코멘트:
이 스크립트는 실제로 복잡한 CLIP 모델을 로드하여 추론하는 대신,
데이터셋에 저장된 '캡션'과 '이미지 설명'이라는 텍스트 능력을 활용하여
AI가 얼마나 똑똑하게 '무엇'을 이해하는지 **시뮬레이션**해보는 것이 핵심 목표입니다!
이 원리를 이해하면, 복잡한 AI도 결국 '말로 설명하는 능력'에서 시작함을 알게 될 거예요!
"""

import random
from datasets import load_dataset
import numpy as np
import pandas as pd
from typing import List, Dict

# --- [설정 상수] ---
DATASET_NAME = "AMANMP0007/military-labeled-clip"
SAMPLE_COUNT = 10 # 실습을 위해 상위 10개 샘플만 사용합니다!

# ==============================================================================
# 🛡️ STEP 1: 데이터셋 로드 및 스트리밍 처리 (Loading the Data)
# ==============================================================================

def load_dataset_safely(dataset_id: str, split: str, sample_k: int):
    """
    스트리밍 모드와 일반 모드를 안전하게 처리하여 데이터셋을 로드합니다.
    (튜터 코멘트: Hugging Face는 크기가 커서 처음엔 스트리밍이 가장 빠르답니다!)
    """
    print("🔍 Step 1: 데이터셋을 안전하게 로드합니다...")
    
    # 1. 스트리밍 모드로 시도 (빠른 탐색에 유리)
    try:
        dataset = load_dataset(dataset_id, split=split, streaming=True)
        print("✨ 성공! 스트리밍 모드로 데이터셋을 로드했습니다. (대용량 데이터에 최적)")
    except Exception as e:
        print(f"⚠️ 경고: 스트리밍 로드 실패 ({e}). 일반(non-streaming) 모드로 전환합니다.")
        # 2. 스트리밍 실패 시, 일반 모드로 로드하여 안전하게 처리
        try:
            dataset = load_dataset(dataset_id, split=split, streaming=False)
            print("✨ 성공! 일반 모드로 데이터셋을 로드했습니다. (작은 크기에 적합)")
        except Exception as e_fallback:
            print(f"❌ 치명적인 오류: 데이터셋 로드 실패. ({e_fallback})")
            return None
    
    # 3. 샘플링된 데이터셋 확보 (Rule 9, 11 적용)
    print(f"✨ 최대 {sample_k}개의 샘플만 가져와 분석하겠습니다.")
    if hasattr(dataset, "take"):
        # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
        # next()를 사용해 이터레이터로 변환합니다.
        sample_iterator = iter(dataset.take(sample_k))
        # 샘플 데이터를 리스트로 변환 (Rule 16, 17 적용)
        sampled_dataset = list(sample_iterator)
    else:
        # 일반 데이터셋 (Dataset)
        sampled_dataset = list(dataset.select(range(min(sample_k, len(dataset)))))

    return sampled_dataset

# ==============================================================================
# 🧠 튜터 AI 분석 시뮬레이션 함수 (The Core Logic)
# ==============================================================================

def analyze_zero_shot_capability(sample: Dict):
    """
    주어진 샘플을 바탕으로 AI가 어떻게 객체를 '이해'하는지 시뮬레이션합니다.
    (튜터 코멘트: 이 부분이 바로 LLM이 작동하는 원리입니다!)
    """
    print("-" * 50)
    print(f"📚 분석 대상 객체: {sample.get('class_name', 'Unknown')}")
    
    # 1. 이미지 캡션 기반 분석 (전체 상황 파악)
    image_cap = sample.get('image_caption', 'No image context provided.')
    print(f"\n🔎 [Contextual Caption 분석 (전체 상황)]: {image_cap[:60]}...")
    
    # 2. 객체 캡션 기반 분석 (무엇을 하고 있는지?)
    object_caption = sample.get('caption', 'No object action provided.')
    print(f"🔬 [Object Action Caption 분석 (객체 행동)]: {object_caption[:60]}...")

    # 3. 제로샷 추론 시뮬레이션 (Zero-Shot Inference Simulation)
    # 만약 AI가 이 정보를 가지고 "이것은 군인이다"라고 결론짓는다면?
    print("\n🤖 [🔥 Zero-Shot Classification 추론 시뮬레이션]")
    if "soldier" in sample.get('class_name', '').lower():
        print("    ✅ AI 추론: 이 객체는 '군인'이다. (이유: 'multi**c**am uniform' 등 복장 정보 포함)")
    elif "tank" in sample.get('class_name', '').lower():
        print("    ✅ AI 추론: 이 객체는 '탱크'이다. (이유: 'armor'나 'large vehicle' 등 특징 정보 포함)")
    else:
        print("    ✅ AI 추론: 캡션 정보와 특징을 종합하여 높은 신뢰도로 해당 클래스로 분류합니다.")

# ==============================================================================
# 🎨 메인 실행 함수
# ==============================================================================

def run_tutorial():
    """
    전체 실습 과정을 진행하는 메인 함수입니다.
    """
    print("==============================================================================")
    print("🚀 Welcome, Future AI Architect! 군사 이미지 분석 튜토리얼을 시작하겠습니다! 🚀")
    print("==============================================================================")
    
    # ----------------------------------------------------------------------
    # 📊 Part 1: 데이터셋 구조 이해 및 기초 통계 분석
    # ----------------------------------------------------------------------
    print("\n\n==============================================================================")
    print("📊 Part 1: 데이터 구조 파헤치기 (Quantitative Analysis)")
    print("==============================================================================")

    # 데이터 로드
    dataset = load_dataset_safely(DATASET_NAME, split='train', sample_k=SAMPLE_COUNT)
    
    if not dataset:
        print("\n🚨 데이터셋 로드 실패로 실습을 진행할 수 없습니다. 스크립트를 종료합니다.")
        return

    # 1. 클래스 분포 분석 (Class Distribution Analysis)
    class_counts = {}
    print(f"\n[🔍 총 {len(dataset)}개의 샘플에서 클래스 분포를 분석합니다]")
    for i, sample in enumerate(dataset):
        class_name = sample.get('class_name')
        if class_name:
            class_counts[class_name] = class_counts.get(class_name, 0) + 1
        
        # 오직 처음 5개의 샘플만 카운트하여 속도 유지 (튜터 코멘트: 데이터가 너무 많을 땐 샘플링이 중요해요!)
        if i >= 5: 
            break
    
    # 결과를 pandas DataFrame으로 보기 좋게 만듭니다.
    df_counts = pd.DataFrame(list(class_counts.items()), columns=['Class Name', 'Count'])
    print("\n📚 데이터셋 클래스별 빈도 분석 (Distribution):")
    print(df_counts.to_string(index=False))
    
    # ----------------------------------------------------------------------
    # 🧠 Part 2: 핵심 AI 원리 실습 - 제로샷 추론 시뮬레이션
    # ----------------------------------------------------------------------
    print("\n\n==============================================================================")
    print(f"🧠 Part 2: AI 핵심 원리 실습! ({SAMPLE_COUNT}개 샘플 탐색)")
    print("==============================================================================")

    # 샘플 반복하며 분석
    print("👉 각 샘플마다 AI가 어떻게 '텍스트'를 이용해 객체를 이해하는지 살펴보겠습니다.")
    
    for i, sample in enumerate(dataset):
        print(f"\n\n=== [ Sample {i+1} / {SAMPLE_COUNT} ] ===")
        # 메인 분석 함수 호출
        analyze_zero_shot_capability(sample)

    # ----------------------------------------------------------------------
    # 🎉 마무리 코멘트
    # ----------------------------------------------------------------------
    print("\n\n==============================================================================")
    print("✨ 튜토리얼 완료! 🎉")
    print("축하합니다! 여러분은 군사 이미지 분석의 핵심 원리인 '캡션 기반의 추론'을 성공적으로 이해했습니다.")
    print("이 데이터셋은 텍스트(캡션)와 이미지(크롭)가 결합되어 있어, AI가 '무엇을 했는지'까지 설명하게 만드는 훌륭한 학습 자료가 된답니다!")
    print("다음 목표: 이 지식을 가지고 실제 CLIP 모델 Fine-Tuning을 시도해보는 것입니다!")
    print("==============================================================================")

if __name__ == "__main__":
    # 주의: 실제 라이브러리 설치가 필요합니다: pip install datasets pandas
    run_tutorial()